# KVO_08 — Satellite prediction of Musanga canopy fraction (five-site benchmark)

**Status:** Research design + executable reference-map audit. Do not fit satellite models until the QA gates pass.

## Scientific questions
1. Can satellite imagery predict local Musanga canopy fraction inferred from frozen high-resolution UAV U-Net maps?
2. Which predictors transfer best between sites: static Sentinel-2 bands/indices, seasonal/multiannual Sentinel-2, or annual satellite embeddings?
3. At which **support scale** (10, 30, 50, 75, 100 m) do models generalize best?
4. Does site-transfer skill differ between disturbed and mature forest, forest structure, and acquisition conditions?

**Five sites:** MOS_1_2, Yoko, Yangambi, Baego, Luki. Luki is entirely outside the UAV training set. The field inventory is independent evaluation for the frozen UAV model, but has only 10 Musanga-positive crowns.

**Key warning:** NB04's original site probability rasters were generated within annotation-derived ROIs. Their NoData is *unknown cover*, not absence of Musanga. For satellite fractions, prefer full-mosaic UAV probability maps or enforce a high and documented valid-UAV coverage fraction. The separate Luki FULL mosaic comes from NB04 Block 09.

**Satellite sources are not downloaded in this version.** First resolve acquisition dates, imagery footprint, coverage, projection, and reference-target validity.

In [ ]:
# 00 — Drive and reference-data contract (CPU only)
from pathlib import Path
import numpy as np
import pandas as pd
import rasterio
from google.colab import drive
MY = Path("/content/drive/MyDrive")
if not MY.exists(): drive.mount("/content/drive")
ROOT = MY / "Colab Notebooks" / "Musanga"
GEO = ROOT / "data" / "Geo_Ref"
KVO04 = ROOT / "KVO_work" / "KVO_04_final_multisite"
MAPS = KVO04 / "probability_maps"
OUT = ROOT / "KVO_work" / "KVO_08_satellite_fraction"
OUT.mkdir(parents=True,exist_ok=True)
RGB = {
    "MOS_1_2": GEO / "MOS_1_2_kvo.tif",
    "Yoko": GEO / "Yoko_kvo_georef.tif",
    "Yangambi": GEO / "Yangambi_georef.tif",
    "Baego": GEO / "Baego_georef.tif",
    "Luki": GEO / "Luki.tif",
}
# Do not fall back silently from full-mosaic to ROI-only probabilities.
PROB = {
    site: MAPS / f"{site}_musanga_probability_uint16_FULL.tif"
    for site in RGB
}
print("NB08 output:",OUT)
print("FULL reference products found:",sum(p.exists() for p in PROB.values()),"/",len(PROB))

In [ ]:
# 01 — Fast metadata audit; never read entire UAV rasters into memory
records=[]
for site, rgb in RGB.items():
    record={"site":site,"rgb_exists":rgb.exists(),
            "full_prob_exists":PROB[site].exists(),
            "roi_prob_exists":(MAPS / f"{site}_musanga_probability_uint16.tif").exists()}
    if rgb.exists():
        with rasterio.open(rgb) as src:
            record.update(crs=str(src.crs),rgb_gsd_x=abs(src.transform.a),
                          rgb_gsd_y=abs(src.transform.e),
                          rgb_width=src.width,rgb_height=src.height,
                          bbox_ha=abs(src.transform.a*src.transform.e)*src.width*src.height/1e4)
    if PROB[site].exists():
        with rasterio.open(PROB[site]) as src:
            record.update(prob_dtype=src.dtypes[0],prob_nodata=src.nodata,
                          probability_scale=src.tags().get("MUSANGA_SCALE",""),
                          map_coverage=src.tags().get("COVERAGE","UNVERIFIED"))
            assert src.dtypes[0]=="uint16" and src.nodata==65535, (site,"encoding mismatch")
    records.append(record)
audit=pd.DataFrame(records)
display(audit)
audit.to_csv(OUT/"site_reference_audit.csv",index=False)
print("STOP GATE: satellite training is disabled until each site's FULL map is available and quality-checked.")

## 02 — Research design and fixed modelling contract

### UAV reference target
- Create one **5 m globally aligned UTM reference grid per site** from *full-coverage* UAV maps, with pixel area weighting where necessary.
- Maintain **valid UAV observation fraction** and **mean Musanga probability** as separate bands. Derive cover-fraction alternatives based on calibrated or fixed U-Net probability thresholds; never treat unmapped pixels as zero.
- Consider crown labels / Luki inventory for independent UAV-prediction calibration and audit. Thresholds must not be optimized using held-out satellite sites.
- Require a minimum observed high-resolution area per satellite grid cell (initial proposal ≥90%; sensitivity: 70%, 80%, 95%).
- Use strictly aligned support grids at 10, 30, 50, 75 and 100 m, all integer multiples of 5 m. Do not interpolate a 10 m satellite measurement into novel 10 m independent observations.

### Satellite feature families (independently ablated)
- **S00** geographic/climatological null baseline only; avoid site identifiers as predictors in site-held-out transfer.
- **S01** Sentinel-2 SR: B2/B3/B4/B5/B6/B7/B8/B8A/B11/B12 and NDVI, EVI, NDRE, NBR, NDMI; use correct resampling of native 20 m bands.
- **S02** Sentinel-2 quality-filtered seasonal percentiles and cloud-free acquisition counts, matched to UAV acquisition epoch.
- **S03** Sentinel-2 monthly/quarterly trajectories, phenological amplitudes and long-term trend; explicitly check missing periods and date leakage.
- **E01** Earth-observation embeddings (e.g. annual Sentinel-2-based embeddings), with provenance, source year and spatial support recorded.
- **F01** spectral + temporal; **F02** embeddings + spectral; **F03** all combined.
- Optional later: Sentinel-1, terrain, forest disturbance history. Keep separate so contribution is identifiable.

### Resolution experiment
Model 10, 30, 50, 75, 100 m *support* with the same site holdouts and comparable coverage gates. Distinguish (i) physical aggregation of UAV fraction, (ii) satellite native spatial resolution, (iii) smoothing/resampling. A 10 m multispectral feature stack contains native 20 m bands and must be labelled accordingly.

### Validation design
**Primary:** five leave-one-site-out folds, training/tuning on four sites; avoid using Luki for any model choice in its held-out fold.
**Secondary:** buffered spatial-block cross-validation within the training sites, blocks substantially larger than the 100 m response grid (test 500 m / 1 km based on autocorrelation).
**No leakage:** tune preprocessing, feature selection, threshold calibration and hyperparameters inside each training fold. Sample weights should avoid dominance by large mosaics and spatially autocorrelated pixels.

### Algorithms
Reference baselines: constant mean, ridge/elastic net and random forest. Main models: HistGradientBoosting or LightGBM plus a regularized embedding model. Optional spatial CNN only after tabular transfer baseline is established. Predict bounded fractions via suitable link/loss or calibration.

### Metrics and science
Primary: site-held-out MAE and RMSE (fraction units), calibration bias and R² relative to within-site variance. Secondary: Spearman ranking, occurrence detection AUC/AP where prevalence is relevant, abundance-stratified error, quality-/coverage-stratified error and bootstrap confidence intervals by spatial block. Report site-by-site and macro averages; do not use random pixel-split scores as external transfer estimates.

### Expected outputs
- Reproducible coverage-checked 5 m UAV reference grids, plus aggregate target grids at all five scales.
- Five-site harmonized spectral/temporal/embedding feature tables and provenance.
- Predeclared experiment matrix and leak-proof fold manifests.
- LO(S)O benchmark, feature ablations, support-scale curves, SHAP/permutation-importance checks on held-out data.
- Final full-predictor model and uncertainty rasters, then ecological tests against disturbance/succession.

In [ ]:
# 03 — Prespecified experiment registry (scaffold; no model fitting)
resolutions=[10,30,50,75,100]
feature_families=["S01_S2_static","S02_S2_seasonal","S03_S2_trajectory",
                  "E01_embeddings","F01_spectral_temporal","F02_embedding_spectral","F03_all"]
algorithms=["ridge","random_forest","hist_gradient_boosting"]
experiments=pd.DataFrame(
    [(r,f,a) for r in resolutions for f in feature_families for a in algorithms],
    columns=["resolution_m","feature_family","algorithm"]
)
experiments["status"]="NOT_RUN"
experiments.to_csv(OUT/"experiment_registry.csv",index=False)
print("Registered experiments:",len(experiments))
print("× 5 independent site holdouts; not fitted until data QA passes.")
display(experiments.head(12))